## Telco Customer Churn Dataser ##
**Objective**: Predict customer retention based on the customer metadata, like services that each customer has signed up for, customer account and demographic information

## 1. Config and Imports

In [2]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
pd.set_option('display.max_columns', None)

In [24]:
DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
BOOLEAN_COLUMNS = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 
                   'PaperlessBilling']

## 2. Helper Functions

### 2.1 Read & Overview

In [4]:
def read_data(path: str) -> pd.DataFrame:
    """Reads the data from the given path and returns a pandas DataFrame."""
    return pd.read_csv(path)

In [29]:
def check_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    """Checks the quality of the data and returns a summary table."""
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")
    print(f"Duplicate rows: {df.duplicated().sum()}\n")
    
    # Create summary table
    summary = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Missing Values': df.isna().sum().values,
        'Missing %': (df.isna().sum().values / len(df) * 100).round(2),
        'Unique Values': df.nunique().values
    })
    
    return summary

In [20]:
def convert_to_boolean(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Converts specified columns in the DataFrame to boolean type."""
    for col in columns:
        df[col] = df[col].map({'Yes': True, 'No': False, 0 : False, 1: True})
    return df

## 3. Read & Overview Data ##
Based on information provided in kaggle dataset description:
* **Unique values**: CustomerID
* **Boolean values**: SeniorCitizen, Partner, Dependents, PhoneService, PaperlessBilling

In [25]:
df = read_data(DATA_PATH)
df = convert_to_boolean(df, BOOLEAN_COLUMNS)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,No,Yes,No,No,No,One year,False,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,No,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,Yes


In [30]:
check_data_quality(df)

Number of rows: 7043
Number of columns: 21
Duplicate rows: 0



,Column,Data Type,Missing Values,Missing %,Unique Values
0,customerID,object,0,0.0,7043
1,gender,object,0,0.0,2
2,SeniorCitizen,bool,0,0.0,2
3,Partner,bool,0,0.0,2
4,Dependents,bool,0,0.0,2
5,tenure,int64,0,0.0,73
6,PhoneService,bool,0,0.0,2
7,MultipleLines,object,0,0.0,3
8,InternetService,object,0,0.0,3
9,OnlineSecurity,object,0,0.0,3


* There **no missing values** in the dataset. 
* **TotalCharges** should be converted to float

In [38]:
df.describe()

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


**Tenure** is number of months for which the subscribtion have been avtive. The minimum value of this column is 0, which may indicates either **missing values** or **accounts which haven't been activated**, if this is rare, these cases should be excluded from further analysis

In [43]:
print(f"Number of cases when tenure is 0: {df[df['tenure'] == 0]['Churn'].shape[0]}")
print(f"Possible Churn Values if tenure is 0: {df[df['tenure'] == 0]['Churn'].unique()}")

Number of cases when tenure is 0: 11
Possible Churn Values if tenure is 0: ['No']


In [44]:
cleaned_df = df[df['tenure'] > 0]
cleaned_df['TotalCharges'] = cleaned_df['TotalCharges'].astype(float)

C:\Users\domin\AppData\Local\Temp\ipykernel_16652\2528438334.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['TotalCharges'] = cleaned_df['TotalCharges'].astype(float)


#### Summary ####
* There's no missing values in the dataset
* For some customers **tenure** is equal 0. There's no information if that indicates missing value or not activated account. Either way is might be misleading for ML, therefore these cases have been excluded from dataset.